In [1]:
# Librerias
import pandas as pd
import numpy as np


In [ ]:
# Carga de los datos (No se si lo vamos a hacer en ingles xd)



RUTA_CR = "../datos/CR.txt"

COLUMNS = [
    "geonameid", "name", "asciiname", "alternatenames", "latitude", "longitude",
    "class", "code", "country", "cc2", "admin1", "admin2", "admin3", "admin4",
    "population", "elevation", "dem", "timezone", "modification_date",
]

def read_geonames(filepath=RUTA_CR):
    df = pd.read_csv(
        filepath, 
        sep="\t", 
        header=None, 
        names=COLUMNS,
    )
    return df

df = read_geonames()
df.head()

,geonameid,name,asciiname,alternatenames,latitude,longitude,class,code,country,cc2,admin1,admin2,admin3,admin4,population,elevation,dem,timezone,modification_date
0,3617025,Río Sábalo,Rio Sabalo,"Rio Sabalo,Rio Sabalos,Río Sábalo,Río Sábalos",11.17604,-85.46081,H,STM,CR,NaN,0.0,NaN,NaN,NaN,0,NaN,34,America/Managua,2026-01-23
1,3617417,Río Orosí,Rio Orosi,"Orosi River,Orosí River,Rio Orosi,Río Orosí",11.14907,-85.34041,H,STM,CR,NaN,0.0,NaN,NaN,NaN,0,NaN,34,America/Managua,2021-05-09
2,3617675,Río Medio Queso,Rio Medio Queso,"Rio Medio Queso,Río Medio Queso",11.09405,-84.64851,H,STM,CR,"CR,NI",0.0,NaN,NaN,NaN,0,NaN,31,America/Managua,2021-09-08
3,3617838,Los Solano,Los Solano,Los Solano,11.06583,-84.70814,P,PPL,CR,NaN,1.0,214.0,21401.0,NaN,0,NaN,29,America/Costa_Rica,2026-05-03
4,3618967,Caño Islas Chicas,Cano Islas Chicas,"Cano Isla Chica,Cano Islas Chicas,Caño Isla Ch...",11.06929,-84.55007,H,STM,CR,NaN,0.0,NaN,NaN,NaN,0,NaN,35,America/Managua,2021-09-08


# Y-opt

El algoritmo Y-opt es una variante del algoritmo 2-opt. Como se vio anteriormente, el algoritmo 2-opt acepta un intercambio de aristas si la suma de las longitudes de las dos aristas nuevas es menor que la suma de las dos aristas viejas; esto se hace comparando únicamente las aristas que se modificaron, ya que el resto de la ruta permanece igual. Por otro lado, el algoritmo Y-opt impone una condición de cambio más estricta, donde, para aceptar un cambio en la ruta, la arista más larga de las nuevas debe ser más corta que la arista más larga de las viejas, y, además, la arista más corta de las nuevas debe ser más corta que la arista más corta de las viejas. Esto hace que el algoritmo Y-opt sea más restrictivo en su condición de aceptación de nuevas rutas, ya que no solo compara la suma total, sino el tamaño individual de cada arista.

La principal ventaja del algoritmo Y-opt es que el número de iteraciones para una ruta de $n$ vértices y $m$ aristas está acotado por $(m-1)n$ (Manthey & van Rhijn, 2025), resultando en un tiempo de ejecución polinomial. En comparación, el algoritmo 2-opt puede necesitar un tiempo exponencial en los peores casos; de esta forma, el algoritmo Y-opt compensa su condición de aceptación con el tiempo que necesitará para ejecutarse. A pesar de que el algoritmo Y-opt sea más estricto, los resultados teóricos muestran que este algoritmo tiene resultados similares a los del 2-opt, donde se llega a rutas ligeramente más largas con el algoritmo Y-opt pero en un tiempo de ejecución acotado en comparación al 2-opt (Manthey & van Rhijn, 2025).

In [ ]:
# Funcion Y-opt

def y_opt(ruta_inicial, matriz_distancias):
    """
    Función para implementar el algoritmo Y-opt
    Entradas :           ruta_inicial ---- Lista con los indices de las ciudades 
                    matriz_distancias ---- Matriz de distancias de tamanyo nxn
                    
    Salidas :             ruta_nueva ---- Lista con la nueva ruta optimizada
    """
    
    n = len(ruta_inicial)
    ruta_copia = ruta_inicial.copy() # Copy para evitar sobreescribir la lista original
    mejorada = True 
    
    while mejorada == True: # Hacer el while hasta que la ruta no se pueda mejorar mas
        mejorada = False # Cambiar la condicion de si se mejoro o no la ruta (Se va a cambiar más adelante)
        
        for i in range(n-1):
            for j in range(i + 2, n): # i + 2 para evitar usar la misma arista en las comparaciones
                if (j == n-1): # Se llega a la última ciudad, la cual debe conectar con la primera siempre
                    k = 0
                else:
                    k = j + 1 # Si no estamos en la última ciudad, la ciudad actual se conecta con la siguiente
               
                # Las 4 ciudades consecutivas son: (Ordenadas segun la ruta inicial)
                a = ruta_copia[i]
                b = ruta_copia[i+1]
                c = ruta_copia[j]
                d = ruta_copia[k]
                
                # Largo de las aristas viejas:
                dist_vieja1 = matriz_distancias[a,b]
                dist_vieja2 = matriz_distancias[c,d]
                
                # Largo de las aristas nuevas:
                dist_nueva1 = matriz_distancias[a,c]
                dist_nueva2 = matriz_distancias[b,d]
                
                # Crear lista con las distancias nuevas
                
                dist_viejas = np.array([dist_vieja1,dist_vieja2])
                dist_nuevas = np.array([dist_nueva1,dist_nueva2])
                
                if (np.max(dist_nuevas) < np.max(dist_viejas)) and (np.min(dist_nuevas) < np.min(dist_viejas)): # Si esto se cumple se puede cambiar la ruta
                    ruta_nueva = ruta_copia[:i+1] + ruta_copia[i+1:j+1][::-1] + ruta_copia[j+1:] # Los + son para concatenar las listas, hay que recordar que en Python las listas empiezan en 0 y se incluye el extremo izquierdo pero no el derecho cuando se extrae una sublista; el [::-1] es para darle vuelta al segmento que está entre las aristas cambiadas
                    ruta_copia = ruta_nueva
                    mejorada = True
                    break # Salir del for de j
            if mejorada == True:
                break  # Salir del for de i
          
    return ruta_copia
                
            
            
            
    